# 🛰️ Live Satellite Tracker Notebook: Overview
## **Real-Time Positions → Ground Tracks → Live Swaths → ROI Countdown**

This notebook gives an **n2yo-style live view** centred on your ROI: where each satellite is now, its ground track for the last 45 and next 90 minutes, its current instrument swath, the day/night terminator, and a countdown to each satellite's next imaging overpass of the ROI, with clocks in **UTC** and the **ROI's local time**.

---

### 🚀 Two Ways to Watch
| Option | Runs where | Best for |
| :--- | :--- | :--- |
| **`live_tracker`** (ipyleaflet) | Inside Jupyter, on Esri World Imagery | Working sessions — updates every 2 s while the notebook stays usable. |
| **`export_live_tracker_html`** | Any browser, no Python | Sharing, GitHub Pages, a second screen. Orbits are propagated in the browser (satellite.js / SGP4). |

> ⚠️ **Note**: ipyleaflet widgets need JupyterLab ≥ 3 or Notebook ≥ 7 with widgets enabled (included in `env.base.yml`). GitHub cannot render live widgets.

## ⚙️ Workspace Configuration, ROI & Orbital Elements
> ### **Same ROI and TLE cache as the prediction notebook**

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "Tools")

from overpass_helper_utility import (
    load_roi,
    resolve_roi_timezone,
    fetch_tles,
    predict_overpasses,
    next_overpasses,
    satellite_positions,
    live_tracker,
    export_live_tracker_html,
)

ROI_GDF = load_roi(Path("DATA") / "SHP" / "Port_Phillip_Bay_Extent_EPSG_32755.shp", name="Port_Phillip_Bay")
ROI_TZ = resolve_roi_timezone(ROI_GDF)
TLES = fetch_tles(tle_dir=Path("DATA") / "TLE")

# Upcoming overpasses feed the countdown column
PASSES = predict_overpasses(ROI_GDF, TLES, days=16, timezone_name=ROI_TZ, verbose=False)
display(next_overpasses(PASSES))

🕒 ROI time zone: Australia/Melbourne (currently UTC+11:00, AEDT)
✔ LANDSAT 8    NORAD 39084  epoch 2026-10-24 06:16 UTC ( 0.7 d old) — cache (0.7 h old)
✔ LANDSAT 9    NORAD 49260  epoch 2026-10-24 03:48 UTC ( 0.9 d old) — cache (0.7 h old)
✔ SENTINEL-2A  NORAD 40697  epoch 2026-10-24 07:19 UTC ( 0.7 d old) — cache (0.7 h old)
✔ SENTINEL-2B  NORAD 42063  epoch 2026-10-24 07:09 UTC ( 0.7 d old) — cache (0.7 h old)
✔ SENTINEL-2C  NORAD 60989  epoch 2026-10-24 04:38 UTC ( 0.8 d old) — cache (0.7 h old)


,Satellite,Countdown,Local,UTC,ROI cover %
0,SENTINEL-2B,1d 00h 12m,Mon 26 Oct 11:27:29,2026-10-26 00:27:29,100.0
1,SENTINEL-2C,3d 00h 02m,Wed 28 Oct 11:17:31,2026-10-28 00:17:31,55.3
2,LANDSAT 9,3d 23h 54m,Thu 29 Oct 11:09:50,2026-10-29 00:09:50,100.0
3,LANDSAT 8,4d 23h 48m,Fri 30 Oct 11:03:47,2026-10-30 00:03:47,65.9
4,SENTINEL-2A,5d 00h 02m,Fri 30 Oct 11:17:39,2026-10-30 00:17:39,54.4


## 📍 Current Positions (Snapshot)
A one-off table of sub-satellite position, altitude, ground speed and direction — the same numbers the live panel refreshes.

In [2]:
display(satellite_positions(TLES))

,satellite,lat,lon,alt_km,ground_speed_kms,direction
0,LANDSAT 8,-32.316971,-25.478855,715.225936,6.815525,Ascending
1,LANDSAT 9,32.244156,154.518507,706.301421,6.832495,Descending
2,SENTINEL-2A,29.302293,-31.059775,792.283752,6.714121,Ascending
3,SENTINEL-2B,64.752970,-44.819482,799.388947,6.702459,Ascending
4,SENTINEL-2C,-64.908539,135.058502,814.287131,6.674946,Descending


## 📡 Live Tracker (ipyleaflet)
> ### **Async Update Loop → Map Layers → Telemetry Panel**

---

### 📂 Map Layers
| Layer | Meaning |
| :--- | :--- |
| **Dot** | Current sub-satellite point. |
| **Dashed / solid line** | Ground track, last 45 min / next 90 min (`track_minutes`). |
| **Shaded strip** | Instrument swath for ±2.5 min around now. When it covers the red ROI, the ROI is being imaged. |
| **Dark overlay** | Night side of the terminator. |

Updates run as an `asyncio` task on the kernel's event loop, so other cells keep working. Use the **Live** toggle, or run the stop cell.

<div class="alert alert-block alert-warning" style="background-color:#fff3cd; color:#664d03; border:1px solid #ffe69c; border-left:6px solid #f0ad4e; border-radius:4px; padding:10px 16px; margin:6px 0;">
<b>⚠️ CAUTION — Live map display</b>
<ul style="margin:6px 0 0 0;">
  <li>The map needs <code>ipywidgets</code> and <code>ipyleaflet</code> in the environment that <b>launched JupyterLab</b>, not only in the kernel. Simplest: <code>conda activate overpass</code> → <code>jupyter lab</code>.</li>
  <li>If you see a model ID or <i>Error displaying widget</i> instead of a map, that is the cause — the standalone page below works regardless.</li>
  <li>Positions come from the loaded TLEs; they are as current as the last <code>fetch_tles()</code> call.</li>
</ul>
</div>

In [3]:
TRACKER = live_tracker(TLES, ROI_GDF, passes=PASSES, refresh_s=2, track_minutes=(45, 90), zoom=3)
TRACKER

In [4]:
# Stop live updates (restart with TRACKER.start())
TRACKER.stop()

## 🌐 Standalone Live-Tracker Page
> ### **Embedded TLEs + Predictions → Browser-Side SGP4 → Shareable HTML**

Writes `OUTPUT/live_tracker.html`, a single page that keeps tracking in real time in any browser. It embeds the current TLEs, the ROI, the predicted overpasses and a Natural Earth coastline, and loads only `satellite.js` from a CDN. Drag to pan, scroll to zoom, switch between **World** and **ROI region**, and click a satellite to hide or show it.

> 💡 **Tip**: The embedded TLEs age. The page footer shows TLE age per satellite; re-export after refreshing TLEs.

<div class="alert alert-block alert-warning" style="background-color:#fff3cd; color:#664d03; border:1px solid #ffe69c; border-left:6px solid #f0ad4e; border-radius:4px; padding:10px 16px; margin:6px 0;">
<b>⚠️ CAUTION — Standalone page ages</b>
<ul style="margin:6px 0 0 0;">
  <li>The page <b>embeds today's TLEs</b>: positions drift as they age (seconds within a week, noticeably after a few weeks). The footer turns amber after 7 days.</li>
  <li>The countdown list ends when the prediction window ends (16 days by default), after which it reports no remaining overpasses.</li>
  <li><b>Re-export after refreshing TLEs</b> to keep a shared copy current.</li>
</ul>
</div>

In [5]:
html_path = export_live_tracker_html(TLES, ROI_GDF, Path("OUTPUT") / "live_tracker.html", passes=PASSES)

🌐 Live tracker written to OUTPUT\live_tracker.html — open it in any browser.
